In [0]:
%pip install openpyxl

In [0]:
# ================================================================
# Schritt 1: Dateien und vorhandene Funktionen prüfen
# ================================================================
import os
import sys

# Ordner, in dem das Notebook ausgeführt wird
basis = "/Workspace/Users/khalil-said.albert@de.abb.com/Daten"
print("Notebook-Ordner:", basis)

# Alle Dateien im aktuellen Ordner einlesen
dateien = sorted(os.listdir(basis))

# Benötigte Python-Dateien prüfen
for name in ["bom_core.py", "layer2_profile.py"]:
    vorhanden = name in dateien
    print(f"{name}: {'GEFUNDEN' if vorhanden else 'FEHLT'}")
    assert vorhanden, f"{name} wurde im Notebook-Ordner nicht gefunden"

# Gezielt nach der korrigierten BOM suchen
bom_kandidaten = [
    name for name in dateien
    if name.lower().endswith("_korrigiert.xlsx")
]

print("Korrigierte BOM-Dateien gefunden:", len(bom_kandidaten))
for name in bom_kandidaten:
    print(" -", name)

assert len(bom_kandidaten) == 1, (
    "Erwartet wird genau eine Excel-Datei mit dem Ende "
    "'_korrigiert.xlsx'."
)

arbeitsdatei = os.path.join(basis, bom_kandidaten[0])

# Aktuellen Ordner für Python-Imports verfügbar machen
if basis not in sys.path:
    sys.path.insert(0, basis)

# Bestehenden Projektcode importieren
import bom_core as bc
import layer2_profile as L2

# Prüfen, ob die benötigten Funktionen wirklich vorhanden sind
assert callable(getattr(bc, "clean_bom", None)), (
    "bom_core.py enthält keine aufrufbare Funktion clean_bom"
)
assert callable(getattr(L2, "baue", None)), (
    "layer2_profile.py enthält keine aufrufbare Funktion baue"
)

print("\nImport erfolgreich.")
print("clean_bom() vorhanden:", callable(bc.clean_bom))
print("L2.baue() vorhanden:", callable(L2.baue))
print("BOM-Pfad vorbereitet:", arbeitsdatei)
print("\nEs wurde noch keine BOM bereinigt und kein Modell gestartet.")

In [0]:
# ================================================================
# Schritt 2: BOM mit dem bestehenden Cleaner bereinigen
# ================================================================
import time
from collections import defaultdict

start = time.time()

# Hier passiert die tatsächliche Bereinigung der Excel-BOM
sauber, protokoll = bc.clean_bom(arbeitsdatei)

dauer = time.time() - start

print(f"Cleaner beendet nach {dauer:.1f} Sekunden")
print("Typ von sauber:", type(sauber).__name__)
print("Bereinigte Positionszeilen:", len(sauber))

# Nur neutrale Kontrollzahlen berechnen – keine Namen ausgeben
produkte = {
    str(zeile["erzeugnis"])
    for zeile in sauber
}

komponenten = {
    str(zeile["komponente"])
    for zeile in sauber
}

bloecke_je_produkt = defaultdict(set)

for zeile in sauber:
    bloecke_je_produkt[str(zeile["erzeugnis"])].add(zeile["block"])

anzahl_bloecke = sum(
    len(bloecke)
    for bloecke in bloecke_je_produkt.values()
)

produkte_mit_mehreren_bloecken = sum(
    len(bloecke) > 1
    for bloecke in bloecke_je_produkt.values()
)

print("Produkte:", len(produkte))
print("Produktblöcke:", anzahl_bloecke)
print("Produkte mit mehreren Blöcken:", produkte_mit_mehreren_bloecken)
print("Unterschiedliche Komponenten:", len(komponenten))

# Erwartete Werte des bestehenden, bereits bekannten Clean-Stands
erwartet = {
    "Positionszeilen": (len(sauber), 47390),
    "Produkte": (len(produkte), 1692),
    "Produktblöcke": (anzahl_bloecke, 1822),
    "Produkte mit mehreren Blöcken": (
        produkte_mit_mehreren_bloecken, 130
    ),
    "Unterschiedliche Komponenten": (len(komponenten), 157),
}

print("\nKontrollvergleich:")
alles_ok = True

for kennzahl, (ist, soll) in erwartet.items():
    status = "OK" if ist == soll else "ABWEICHUNG"
    alles_ok = alles_ok and ist == soll
    print(f"{kennzahl}: {ist} | erwartet {soll} | {status}")

print("\nGesamtergebnis:", "CLEAN-STAND REPRODUZIERT" if alles_ok
      else "STOPP – ABWEICHUNG UNTERSUCHEN")

In [0]:
# ================================================================
# Schritt 3: Produkt-Komponenten-Daten aus der sauberen BOM bilden
# ================================================================
import pandas as pd

# Jede bereinigte Stücklistenposition wird zunächst übernommen.
# Hauptgruppe und kleinere Gruppe werden mit den bestehenden
# freigegebenen Funktionen bestimmt.
produkt_komponente = pd.DataFrame([
    {
        "produkt": str(zeile["erzeugnis"]),
        "hauptgruppe": L2.hauptgruppe_von(zeile["erzeugnis"]),
        "kleinere_gruppe": L2.typ_von(zeile["erzeugnis_txt"]),
        "komponente": str(zeile["komponente"]),
    }
    for zeile in sauber
])

print("Positionszeilen vor Zusammenfassung:",
      len(produkt_komponente))

# Prüfen, ob jedes Produkt eindeutig genau einer Hauptgruppe
# und genau einer kleineren Gruppe zugeordnet wird.
zuordnungen = (
    produkt_komponente
    .groupby("produkt")[["hauptgruppe", "kleinere_gruppe"]]
    .nunique()
)

mehrdeutige_hauptgruppen = int(
    (zuordnungen["hauptgruppe"] > 1).sum()
)

mehrdeutige_kleinere_gruppen = int(
    (zuordnungen["kleinere_gruppe"] > 1).sum()
)

print("Produkte mit mehreren Hauptgruppen:",
      mehrdeutige_hauptgruppen)
print("Produkte mit mehreren kleineren Gruppen:",
      mehrdeutige_kleinere_gruppen)

assert mehrdeutige_hauptgruppen == 0
assert mehrdeutige_kleinere_gruppen == 0

# Dieselbe Komponente darf pro Produkt nur einmal vorkommen.
# Mehrere Blöcke oder Positionszeilen erhöhen ihr Gewicht nicht.
produkt_komponente = (
    produkt_komponente
    .drop_duplicates(["produkt", "komponente"])
    .reset_index(drop=True)
)

# Nur neutrale Kontrollzahlen ausgeben
anzahl_produkte = produkt_komponente["produkt"].nunique()
anzahl_hauptgruppen = produkt_komponente["hauptgruppe"].nunique()
anzahl_kleinere_gruppen = (
    produkt_komponente[
        ["hauptgruppe", "kleinere_gruppe"]
    ]
    .drop_duplicates()
    .shape[0]
)
anzahl_komponenten = produkt_komponente["komponente"].nunique()

print("\nNach einmaliger Zählung je Produkt und Komponente:")
print("Produkt-Komponenten-Kombinationen:",
      len(produkt_komponente))
print("Produkte:", anzahl_produkte)
print("Hauptgruppen:", anzahl_hauptgruppen)
print("Kleinere Gruppen:", anzahl_kleinere_gruppen)
print("Unterschiedliche Komponenten:", anzahl_komponenten)

print("\nKontrollvergleich:")
print("Produkte:", "OK" if anzahl_produkte == 1692 else "ABWEICHUNG")
print("Hauptgruppen:", "OK" if anzahl_hauptgruppen == 4 else "ABWEICHUNG")
print("Kleinere Gruppen:",
      "OK" if anzahl_kleinere_gruppen == 37 else "ABWEICHUNG")
print("Komponenten:",
      "OK" if anzahl_komponenten == 157 else "ABWEICHUNG")

In [0]:
# ================================================================
# Schritt 4: Vollständige Produkt-Komponenten-Matrix
# ================================================================

# Eindeutige Gruppenzuordnung jedes Produkts
produkt_info_alle = (
    produkt_komponente[
        ["produkt", "hauptgruppe", "kleinere_gruppe"]
    ]
    .drop_duplicates()
    .set_index("produkt")
)

assert not produkt_info_alle.index.duplicated().any()

# Zeile = Produkt
# Spalte = Komponente
# 1 = im Produkt vorhanden
# 0 = im Produkt nicht vorhanden
X_produkte_alle = pd.crosstab(
    index=produkt_komponente["produkt"],
    columns=produkt_komponente["komponente"]
)

X_produkte_alle = (X_produkte_alle > 0).astype("int8")

# Gruppenzuordnungen in dieselbe Reihenfolge wie die Matrix bringen
produkt_info_alle = produkt_info_alle.loc[X_produkte_alle.index]

belegte_zellen = int(X_produkte_alle.values.sum())
alle_zellen = int(
    X_produkte_alle.shape[0] * X_produkte_alle.shape[1]
)
anteil_vorhanden = belegte_zellen / alle_zellen

anzahl_kleinere_gruppen = (
    produkt_info_alle[
        ["hauptgruppe", "kleinere_gruppe"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("Vollständige Modelleingabe:")
print("Produkte:", X_produkte_alle.shape[0])
print("Komponenten:", X_produkte_alle.shape[1])
print("Matrixform:", X_produkte_alle.shape)
print("Kleinere Gruppen:", anzahl_kleinere_gruppen)
print("Vorhandene Produkt-Komponenten-Verbindungen:",
      belegte_zellen)
print(f"Anteil vorhandener Verbindungen: {anteil_vorhanden:.3%}")

print("\nKontrollvergleich:")
print("Produkte:",
      "OK" if X_produkte_alle.shape[0] == 1692 else "ABWEICHUNG")
print("Komponenten:",
      "OK" if X_produkte_alle.shape[1] == 157 else "ABWEICHUNG")
print("Kleinere Gruppen:",
      "OK" if anzahl_kleinere_gruppen == 37 else "ABWEICHUNG")
print("Verbindungen:",
      "OK" if belegte_zellen == 38654 else "ABWEICHUNG")

print("\nKeine Produkte und keine kleineren Gruppen ausgeschlossen.")


In [0]:
# ================================================================
# Schritt 5: Erster NMF-Pilot nur für Hauptgruppe GJL121
# ================================================================
import time
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF

ZIEL_HAUPTGRUPPE = "GJL121"

# Produkte der ausgewählten Hauptgruppe bestimmen
ziel_produkte = produkt_info_alle.index[
    produkt_info_alle["hauptgruppe"] == ZIEL_HAUPTGRUPPE
]

# Nur ihre Produkt-Komponenten-Zeilen auswählen
X_ziel = X_produkte_alle.loc[ziel_produkte].copy()

# Komponenten entfernen, die in dieser Hauptgruppe kein einziges Mal
# vorkommen. Diese Spalten bestehen nur aus Nullen und enthalten
# für dieses Modell keine Information.
X_ziel = X_ziel.loc[:, X_ziel.sum(axis=0) > 0]

# Gruppenzuordnungen werden separat aufbewahrt.
# Sie gehen NICHT in das Modell ein.
ziel_info = produkt_info_alle.loc[X_ziel.index].copy()

print("Pilot-Hauptgruppe:", ZIEL_HAUPTGRUPPE)
print("Produkte im Modell:", X_ziel.shape[0])
print("Komponenten im Modell:", X_ziel.shape[1])
print(
    "Kleinere Gruppen vorhanden:",
    ziel_info["kleinere_gruppe"].nunique()
)

# Wir wissen noch nicht, wie viele Komponentenmustern sinnvoll sind.
muster_anzahlen = [2, 3, 4, 5, 6, 8, 10, 12, 15]

nmf_modelle = {}
nmf_ergebnisse = []

for anzahl_muster in muster_anzahlen:
    start = time.time()

    modell = NMF(
        n_components=anzahl_muster,
        init="nndsvda",
        random_state=20260908,
        max_iter=1000
    )

    # Hier beginnt das eigentliche Lernen.
    W = modell.fit_transform(X_ziel.values)
    H = modell.components_

    rekonstruktion = W @ H

    relativer_fehler = (
        np.linalg.norm(X_ziel.values - rekonstruktion)
        / np.linalg.norm(X_ziel.values)
    )

    dauer = time.time() - start

    nmf_modelle[anzahl_muster] = {
        "modell": modell,
        "W": W,
        "H": H,
        "produkte": list(X_ziel.index),
        "komponenten": list(X_ziel.columns),
    }

    nmf_ergebnisse.append({
        "anzahl_muster": anzahl_muster,
        "relativer_fehler": relativer_fehler,
        "durchlaeufe": int(modell.n_iter_),
        "dauer_sekunden": dauer,
    })

    print(
        f"{anzahl_muster:>2} Muster | "
        f"Fehler {relativer_fehler:.4f} | "
        f"{modell.n_iter_:>4} Durchläufe | "
        f"{dauer:.2f} Sekunden"
    )

nmf_ergebnisse = pd.DataFrame(nmf_ergebnisse)

print("\nPilotläufe abgeschlossen:", len(nmf_ergebnisse))
print("B6/B6S wurde beim Lernen nicht verwendet.")

In [0]:
# ================================================================
# Schritt 6: Gelernte Komponentenmustern sichtbar machen
# ================================================================

GEZEIGTE_MUSTERANZAHL = 6
TOP_KOMPONENTEN = 10

ansicht = nmf_modelle[GEZEIGTE_MUSTERANZAHL]
H = ansicht["H"]
komponenten_reihenfolge = ansicht["komponenten"]

# Häufigsten vorhandenen Kurztext je Komponente bestimmen.
# Der Kurztext wird ausschließlich angezeigt.
# Er wurde nicht zum Trainieren des Modells verwendet.
komponenten_texte = pd.DataFrame([
    {
        "komponente": str(zeile["komponente"]),
        "kurztext": str(zeile["komponente_txt"] or "").strip()
    }
    for zeile in sauber
])

komponenten_texte = komponenten_texte[
    komponenten_texte["kurztext"] != ""
]

text_haeufigkeit = (
    komponenten_texte
    .groupby(["komponente", "kurztext"])
    .size()
    .reset_index(name="vorkommen")
    .sort_values(
        ["komponente", "vorkommen"],
        ascending=[True, False]
    )
    .drop_duplicates("komponente")
    .set_index("komponente")["kurztext"]
    .to_dict()
)

muster_anzeige = []

for muster_index in range(H.shape[0]):
    gewichte = H[muster_index]

    # Komponenten mit dem höchsten Gewicht in diesem Muster
    top_positionen = np.argsort(gewichte)[::-1][:TOP_KOMPONENTEN]

    hoechstes_gewicht = gewichte[top_positionen[0]]

    for rang, position in enumerate(top_positionen, start=1):
        komponente = komponenten_reihenfolge[position]
        gewicht = gewichte[position]

        muster_anzeige.append({
            "muster": f"Muster {muster_index + 1}",
            "rang_im_muster": rang,
            "komponente": komponente,
            "kurztext_nur_anzeige":
                text_haeufigkeit.get(komponente, ""),
            "gewicht": gewicht,
            "relativ_zum_staerksten":
                gewicht / hoechstes_gewicht
                if hoechstes_gewicht > 0 else 0.0
        })

muster_anzeige = pd.DataFrame(muster_anzeige)

display(
    muster_anzeige.sort_values(
        ["muster", "rang_im_muster"]
    )
)

In [0]:
# ================================================================
# Schritt 7: Ähnlichkeit der kleineren Gruppen im NMF-Modell
# ================================================================
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

GEZEIGTE_MUSTERANZAHL = 5
ansicht = nmf_modelle[GEZEIGTE_MUSTERANZAHL]

# W als lesbare Tabelle:
# eine Zeile je Produkt, eine Spalte je gelerntem Muster
W_produkte = pd.DataFrame(
    ansicht["W"],
    index=ansicht["produkte"],
    columns=[
        f"Muster_{i + 1}"
        for i in range(GEZEIGTE_MUSTERANZAHL)
    ]
)

# Musterwerte jedes Produkts relativieren.
# Dadurch zählt jedes Produkt gleich stark, unabhängig davon,
# wie groß seine absoluten NMF-Werte sind.
zeilensummen = W_produkte.sum(axis=1).replace(0, np.nan)

W_relativ = (
    W_produkte
    .div(zeilensummen, axis=0)
    .fillna(0.0)
)

# Kleinere Gruppe jedes Produkts ergänzen
W_mit_gruppe = W_relativ.copy()
W_mit_gruppe["kleinere_gruppe"] = (
    ziel_info.loc[W_relativ.index, "kleinere_gruppe"]
)

# Durchschnittliches Musterprofil je kleinerer Gruppe
gruppen_muster = (
    W_mit_gruppe
    .groupby("kleinere_gruppe")
    .mean()
)

# Produktzahl je kleinerer Gruppe
produkte_je_gruppe = (
    W_mit_gruppe["kleinere_gruppe"]
    .value_counts()
)

print(
    f"Gruppenprofile aus dem Modell mit "
    f"{GEZEIGTE_MUSTERANZAHL} Mustern:"
)

gruppen_muster_anzeige = gruppen_muster.copy()
gruppen_muster_anzeige.insert(
    0,
    "produkte_N",
    produkte_je_gruppe.loc[gruppen_muster.index]
)

display(
    gruppen_muster_anzeige
    .reset_index()
    .round(4)
)

# Ähnlichkeit zwischen allen kleineren Gruppen berechnen
gruppen_namen = list(gruppen_muster.index)

S_gruppen = cosine_similarity(gruppen_muster.values)

paar_ergebnisse = []

for i, j in combinations(range(len(gruppen_namen)), 2):
    gruppe_a = gruppen_namen[i]
    gruppe_b = gruppen_namen[j]

    n_a = int(produkte_je_gruppe[gruppe_a])
    n_b = int(produkte_je_gruppe[gruppe_b])

    paar_ergebnisse.append({
        "gruppe_a": gruppe_a,
        "gruppe_b": gruppe_b,
        "produkte_a": n_a,
        "produkte_b": n_b,
        "nmf_aehnlichkeit": float(S_gruppen[i, j]),
        "hinweis":
            "mindestens eine Gruppe hat weniger als 3 Produkte"
            if min(n_a, n_b) < 3
            else ""
    })

paar_ergebnisse = (
    pd.DataFrame(paar_ergebnisse)
    .sort_values(
        "nmf_aehnlichkeit",
        ascending=False
    )
    .reset_index(drop=True)
)

paar_ergebnisse.insert(
    0,
    "rang",
    np.arange(1, len(paar_ergebnisse) + 1)
)

print(
    f"\nAlle {len(paar_ergebnisse)} Gruppenpaare, "
    "nach NMF-Ähnlichkeit sortiert:"
)

display(
    paar_ergebnisse.round({
        "nmf_aehnlichkeit": 4
    })
)

In [0]:
n_muster = ansicht["W"].shape[1]
muster_namen = [
    f"Muster_{i + 1}"
    for i in range(n_muster)
]

W_beschriftet = pd.DataFrame(
    ansicht["W"],
    index=ansicht["produkte"],
    columns=muster_namen
)

H_beschriftet = pd.DataFrame(
    ansicht["H"],
    index=muster_namen,
    columns=ansicht["komponenten"]
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None
):
    print("W: Produkte × Muster")
    display(W_beschriftet.round(4))

    print("H: Muster × Komponenten")
    display(H_beschriftet.round(4))